# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [50]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [51]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [52]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [53]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [54]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [55]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [56]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [57]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [58]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [59]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [60]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48


### La soumission


In [61]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [62]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [63]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "https://github.com/Precieux-Stack/Mon-projet"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 19:28:48"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 89936 lignes sur 149894


### L'interface, telle qu'on l'appelle


In [64]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.51 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [65]:
import numpy as np   # pour fabriquer le tableau de 0/1 du prédicteur

# ── 1. Part d'absents sur les lignes de contrôle, en UNE ligne, en pourcentage.
# yval vaut 1 si le patient est absent : sa moyenne EST la part d'absents.
part_absents_controle = 100 * yval.mean()

# ── 2. La RÉFÉRENCE sur les lignes de contrôle : une EXACTITUDE, pas une part d'absents.
# C'est le score d'un modèle qui répond toujours la classe majoritaire ("le patient vient" = 0).
classe_majoritaire = yval.mode()[0]                              # la classe qui revient le plus souvent
ma_reference = 100 * (yval == classe_majoritaire).mean()         # part de lignes bien classées par ce modèle

# ── 3. Le prédicteur : un tableau de 0/1 de la même longueur que yval, rempli de la classe majoritaire.
pred = np.full(len(yval), classe_majoritaire).astype(int)

# Preuve qu'il ATTEINT ma_reference (pas "environ") : on calcule son score et on compare.
score_pred = 100 * (pred == yval.to_numpy()).mean()
assert abs(score_pred - ma_reference) < 1e-9, "le prédicteur doit coïncider avec ma_reference"
print(f"part d'absents (contrôle) : {part_absents_controle:.2f} %")
print(f"ma_reference (contrôle)   : {ma_reference:.2f} %  | score du prédicteur : {score_pred:.2f} %")

# ── 4. L'écart entre ma référence (contrôle) et `reference` (tout train), en points.
ecart_a_train = ma_reference - reference
print(f"ecart_a_train : {ecart_a_train:+.3f} point(s)")
# Pourquoi ce petit écart compte : deux modèles évalués sur des jeux différents n'ont pas la même
# référence. Comparer un score mesuré sur le contrôle à la référence de tout train, c'est mélanger
# deux bases de comparaison : un écart de quelques centièmes pourrait être pris pour un gain.

# ── 5. Le collègue qui annonce 83 %.
score_collegue = 83.0
ecart_collegue = score_collegue - ma_reference                   # écart à la référence, en points
ecart_en_barres = ecart_collegue / erreur_type                   # la barre de bruit du cours : combien de barres ?

reponse_au_collegue = (
    f"Ton 83 % dépasse la référence de {ecart_collegue:+.2f} point(s), soit {ecart_en_barres:.1f} "
    f"barre(s) de bruit (barre = {erreur_type:.2f} point). "
    + ("C'est plus de deux barres : ce n'est pas du bruit, mais il faut vérifier le jeu sur lequel il a mesuré."
       if abs(ecart_en_barres) >= 2 else
       "C'est moins de deux barres : ce n'est pas un résultat, juste du bruit de découpage.")
)
print(reponse_au_collegue)

controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


part d'absents (contrôle) : 18.25 %
ma_reference (contrôle)   : 81.75 %  | score du prédicteur : 81.75 %
ecart_a_train : +0.001 point(s)
Ton 83 % dépasse la référence de +1.25 point(s), soit 5.7 barre(s) de bruit (barre = 0.22 point). C'est plus de deux barres : ce n'est pas du bruit, mais il faut vérifier le jeu sur lequel il a mesuré.
  ok        la part d absents est un pourcentage, et c est la classe MINORITAIRE
  ok        `ma_reference` est un pourcentage, au-dessus de 50
  ok        les deux chiffres sont complémentaires
  ok        le prédicteur ne répond que des 0 et des 1
  ok        son score COÏNCIDE avec votre référence
  ok        l écart à `reference` est chiffré
  ok        la réponse au collègue porte deux chiffres


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [66]:
# ── L'OUTIL : balayer toutes les colonnes d'une table ──────────────────────────────
# Pour chaque colonne : on groupe les lignes par valeur de cette colonne, on calcule le
# taux d'absence de chaque groupe (la moyenne de la cible), puis l'écart entre le taux
# le plus haut et le taux le plus bas, en POINTS (0 à 100).
def balayer(df, taille_min=1):
    """taille_min=1 : aucun filtre. taille_min=500 : on ne garde que les groupes >= 500 lignes."""
    lignes = []
    for col in df.columns:
        if col == CIBLE:                      # on ne balaie pas la cible par elle-même
            continue
        # Pour chaque valeur de la colonne : le taux d'absence ("mean") et l'effectif ("size")
        g = df.groupby(col)[CIBLE].agg(taux="mean", effectif="size")
        # Le critère : on retire les groupes trop petits AVANT de comparer
        g = g[g["effectif"] >= taille_min]
        if g.empty:                           # aucun groupe assez gros : la colonne disparaît
            continue
        lignes.append({
            "colonne": col,
            "ecart": 100 * (g["taux"].max() - g["taux"].min()),   # en points
            "nb_groupes": len(g),
            "plus_petit_groupe": int(g["effectif"].min()),
        })
    # On trie du plus grand écart au plus petit
    return pd.DataFrame(lignes).set_index("colonne").sort_values("ecart", ascending=False)

# ── 1. Balayage SANS filtre, sur VOTRE échantillon (mon_train) ─────────────────────
balayage_brut = balayer(mon_train, taille_min=1)
display(balayage_brut.head(8))

# ── 2. La colonne gagnante du balayage brut : le piège ─────────────────────────────
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = balayage_brut.iloc[0]["ecart"]
# Combien de ses groupes atteignent 500 lignes ? (attendu : 0, chaque ligne a son propre groupe)
groupes_de_500 = int((mon_train.groupby(colonne_piegee).size() >= 500).sum())
print(f"colonne piégée : {colonne_piegee} | écart : {ecart_non_filtre:.1f} points | groupes de 500 : {groupes_de_500}")

# ── 3. Balayage FILTRÉ : seulement les groupes d'au moins 500 lignes, PUIS on compare ─
classement = balayer(mon_train, taille_min=500)
display(classement.head(8))
plus_petit_groupe_du_classement = int(classement["plus_petit_groupe"].min())

# ── 4. La meilleure colonne après filtre ───────────────────────────────────────────
meilleure_colonne = classement.index[0]
ecart_filtre = classement.iloc[0]["ecart"]
print(f"meilleure colonne : {meilleure_colonne} | écart : {ecart_filtre:.1f} points")

# Repère pour la lecture : le bruit d'un groupe de 500 lignes (erreur-type, en points)
p = mon_train[CIBLE].mean()
barre_groupe_500 = 100 * (p * (1 - p) / 500) ** 0.5
print(f"bruit d'un groupe de 500 lignes : ± {barre_groupe_500:.2f} point(s)")

# ── Les deux textes : À RELIRE et à reformuler avec tes mots après avoir vu tes résultats ──
ce_que_le_critere_ne_protege_pas = (
    f"En tête du classement filtré : {meilleure_colonne} ({ecart_filtre:.1f} points), sur des groupes "
    f"d'au moins 500 lignes. Le critère ne vérifie que la TAILLE des groupes, pas la nature de la colonne : "
    f"il faut vérifier en plus si cette colonne serait connue AU MOMENT de décider, "
    f"ou seulement après l'issue du rendez-vous (fuite d'information)."
)
lecture = (
    f"{colonne_piegee} gagnait parce que chaque valeur y est quasi unique : un groupe d'une ligne a un "
    f"taux de 0 % ou de 100 %, donc l'écart atteint {ecart_non_filtre:.0f} points par construction. "
    f"Par rapport au bruit d'un groupe (± {barre_groupe_500:.2f} point pour 500 lignes, bien plus pour "
    f"1 ligne), cet écart ne prouve rien."
)

controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


,ecart,nb_groupes,plus_petit_groupe
colonne,,,
dossier_id,100.0,89936,1
ligne_id,100.0,89936,1
patient_id,100.0,68423,1
indice_risque_externe,100.0,8141,1
taille_cm,100.0,646,1
date_prise,100.0,10034,1
indice_meteo_jour,100.0,146,1
cout_acte,100.0,1393,1


colonne piégée : dossier_id | écart : 100.0 points | groupes de 500 : 0


,ecart,nb_groupes,plus_petit_groupe
colonne,,,
duree_consultation,99.378238,30,524
canal_prise,42.299989,4,11611
taux_absence_passe,28.478321,3,1007
delai_jours,25.060205,7,4736
clinique,21.969502,80,912
quartier,20.820941,21,561
cout_acte,5.547615,5,502
age,5.490548,64,503


meilleure colonne : duree_consultation | écart : 99.4 points
bruit d'un groupe de 500 lignes : ± 1.72 point(s)
  ok        le balayage couvre toutes les colonnes
  ok        l écart non filtré est de l ordre de 100 points
  ok        la colonne piégée n a AUCUN groupe de 500 lignes
  ok        le classement filtré est plus petit que le non filtré
  ok        la meilleure colonne est bien celle de VOTRE classement
  ok        le classement filtré ne garde QUE des groupes de 500 lignes et plus
  ok        les deux écarts sont chiffrés en POINTS, et le filtré est plus petit
  ok        la lecture compare, et le critère est interrogé


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [67]:
# ── CONSTAT A : la colonne qui gagne après filtre est une colonne "connue après coup" ──
# On lit les chiffres directement dans NOTRE classement (Q2) : on ne recopie rien à la main.
ecart_duree = classement.loc["duree_consultation", "ecart"]    # écart de la colonne suspecte (en points)
ecart_passe = classement.loc["taux_absence_passe", "ecart"]    # meilleure colonne gardée dans la table de travail

constat(
    "la meilleure colonne du classement filtré est connue après l'issue",              # l'observation
    f"duree_consultation : {ecart_duree:.1f} points d'écart entre groupes de 500 lignes et plus",   # le chiffre
    f"contre {ecart_passe:.1f} points pour taux_absence_passe, la meilleure colonne gardée dans la table de travail",  # par rapport à quoi
    "écarter : elle figure dans A_ECARTER, et cette mesure confirme pourquoi (un absent n'a pas de consultation)",    # la décision
)

# ── CONSTAT B (NÉGATIF) : le critère des 500 lignes ne trouve AUCUN groupe pour la colonne piégée ──
nb_valeurs = mon_train[colonne_piegee].nunique()               # nombre de valeurs distinctes de la colonne piégée
nb_lignes = len(mon_train)                                     # nombre de lignes de notre échantillon

constat(
    f"aucun groupe de 500 lignes pour {colonne_piegee}",
    f"0 groupe de 500 lignes sur {nb_valeurs} valeurs distinctes de {colonne_piegee}",
    f"contre au moins 500 lignes exigées par groupe : {nb_lignes} lignes pour {nb_valeurs} valeurs, "
    f"donc environ 1 ligne par valeur",
    "écarter : son écart de 100 points est un artefact, pas un signal",
)

import re

controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la meilleure colonne du classement filtré est ...,duree_consultation : 99.4 points d'écart entre...,"contre 28.5 points pour taux_absence_passe, la...","écarter : elle figure dans A_ECARTER, et cette..."


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la meilleure colonne du classement filtré est ...,duree_consultation : 99.4 points d'écart entre...,"contre 28.5 points pour taux_absence_passe, la...","écarter : elle figure dans A_ECARTER, et cette..."
4,aucun groupe de 500 lignes pour dossier_id,0 groupe de 500 lignes sur 89936 valeurs disti...,contre au moins 500 lignes exigées par groupe ...,écarter : son écart de 100 points est un artef...


  ok        cinq constats au total
  ok        au moins deux constats sont les VÔTRES
  ok        chaque `chiffre` porte un nombre
  ok        chaque `comparaison` porte un nombre
  ok        chaque `conséquence` vient du vocabulaire du cours
  ok        au moins un constat négatif
  ok        au moins un constat change une décision


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [68]:
# ── CELLULE A : tes trois prédictions, AVANT toute mesure (en barres de bruit) ──
# Chaque nombre = de combien de barres tu attends que le score bouge. Remplace-les par ce que TU attends.

# Âge à -1 (arbres) : un arbre coupe sur des seuils ("âge < 30 ?"). Un -1 tombe simplement dans
# le groupe des plus petits âges : il ne casse rien. J'attends un écart d'environ 1 barre ou moins.
pred_age = 1

# Lignes dupliquées : si une même ligne est à la fois en entraînement et en validation, le modèle
# "reconnaît" des lignes vues et le score est gonflé. S'il y en a peu : petit effet.
pred_doublons = 1

# Nombre rangé en texte : si la colonne est déjà écartée (A_ECARTER), la corriger ne change rien.
pred_taille = 0

# On les enregistre avec l'horloge du carnet (c'est la règle de l'ordre du cours).
prevoir("âge à -1 : arbres", f"écart d'environ {pred_age} barre ou moins",
        "un arbre coupe sur des seuils : -1 tombe dans le groupe des petits âges")
prevoir("lignes dupliquées", f"écart d'environ {pred_doublons} barre",
        "peu de doublons : un petit gonflement du score, au plus")
prevoir("nombre rangé en texte", f"écart de {pred_taille} barre",
        "colonne déjà hors de la table de travail : la corriger ne change rien")
import numpy as np

# ── 0. REPÈRES : combien de lignes ont un âge impossible, combien de doublons, quels textes ──
COLS_AGE = [c for c in ["age", "age_mois"] if c in X_sans_features.columns]   # age et age_mois disent la même chose
nb_impossibles = int((X_sans_features["age"] < 0).sum())                      # âges négatifs (le -1 sentinelle)
print(f"lignes avec un âge impossible : {nb_impossibles} sur {len(X_sans_features)}")

nb_doublons = int(train.drop(columns=[ID]).duplicated().sum())               # on ignore l'identifiant (unique)
print(f"lignes dupliquées (hors identifiant) : {nb_doublons}")

colonnes_texte_numeriques = []
for c in train.select_dtypes(include="object").columns:
    lu = pd.to_numeric(train[c], errors="coerce")                             # tente de lire la colonne comme un nombre
    if lu.notna().mean() > 0.9:                                               # plus de 90 % se lisent comme un nombre
        colonnes_texte_numeriques.append(c)
print("colonnes de nombres rangés en texte :", colonnes_texte_numeriques,
      "| déjà écartées :", [c for c in colonnes_texte_numeriques if c in A_ECARTER])

# ── 1. LES ARBRES : avec l'âge tel quel, puis âge corrigé ──
s_avec_age = score_interne                                    # le modèle de la mise en place (âge tel quel)
X_corr = X_sans_features.copy()
for c in COLS_AGE:
    X_corr.loc[X_corr[c] < 0, c] = np.nan                     # âge impossible -> valeur manquante (les arbres savent la traiter)
Xtr_c, Xval_c = X_corr.loc[Xtr.index], X_corr.loc[Xval.index] # EXACTEMENT les mêmes lignes qu'avant
modele_c = HistGradientBoostingClassifier(random_state=0).fit(Xtr_c, ytr)
s_sans_age = 100 * modele_c.score(Xval_c, yval)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type       # l'écart, divisé par la barre de bruit
print(f"arbres : avec {s_avec_age:.2f} % | sans {s_sans_age:.2f} % | écart {age_en_barres:+.2f} barre(s)")

# ── 2. LE MODÈLE LINÉAIRE : la MÊME correction ──
lin_avec_age = score_lineaire                                 # le linéaire de la mise en place (âge tel quel)
X_corr_lin = X_sans_features.copy()
for c in COLS_AGE:
    mediane_valide = X_corr_lin.loc[X_corr_lin[c] >= 0, c].median()      # médiane des âges possibles
    X_corr_lin.loc[X_corr_lin[c] < 0, c] = mediane_valide                # un linéaire n'accepte pas de NaN
Xc_num = pd.get_dummies(X_corr_lin, columns=CAT).fillna(0)               # même préparation que la mise en place
modele_lin_c = LogisticRegression(solver="newton-cholesky").fit(Xc_num.loc[Xtr.index], ytr)
lin_sans_age = 100 * modele_lin_c.score(Xc_num.loc[Xval.index], yval)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type
print(f"linéaire : avec {lin_avec_age:.2f} % | sans {lin_sans_age:.2f} % | écart {lin_age_en_barres:+.2f} barre(s)")

# ── 3. LES VERDICTS : modèles de phrases avec TES chiffres. Relis-les et reformule avec tes mots. ──
pas_de_difference = max(abs(age_en_barres), abs(lin_age_en_barres)) < 2

verdict_age = (
    f"Seulement {nb_impossibles} lignes ont un âge impossible ({100 * nb_impossibles / len(train):.2f} % de la table). "
    f"Corriger l'âge change le score des arbres de {age_en_barres:+.2f} barre(s) et celui du modèle linéaire de "
    f"{lin_age_en_barres:+.2f} barre(s). "
    + ("Aucun des deux n'atteint 2 barres, donc ce n'est pas une différence : en théorie un arbre ignore le -1 "
       "(il coupe sur un seuil) alors qu'une somme pondérée s'en laisse tirer, mais ce défaut est trop rare ici "
       "pour peser sur l'un ou l'autre : il est établi, mais il ne coûte rien."
       if pas_de_difference else
       "Au moins un des deux écarts dépasse 2 barres : la correction change réellement le score de ce modèle.")
)

verdict_doublons = (
    f"Le contrôle trouve {nb_doublons} ligne(s) dupliquée(s) sur {len(train)}. "
    + ("Un contrôle qui ne trouve rien est un résultat : on a cherché, il n'y a rien à corriger, "
       "et on peut le dire avec un chiffre (0 sur " + str(len(train)) + ")."
       if nb_doublons == 0 else
       "Il faut mesurer leur effet avant de décider : un diagnostic certain ne dit pas ce que le défaut coûte.")
)

print(verdict_age)
print(verdict_doublons)

controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55


lignes avec un âge impossible : 46 sur 149894
lignes dupliquées (hors identifiant) : 0
colonnes de nombres rangés en texte : [] | déjà écartées : []
arbres : avec 82.51 % | sans 82.49 % | écart -0.06 barre(s)
linéaire : avec 82.21 % | sans 82.21 % | écart +0.00 barre(s)
Seulement 46 lignes ont un âge impossible (0.03 % de la table). Corriger l'âge change le score des arbres de -0.06 barre(s) et celui du modèle linéaire de +0.00 barre(s). Aucun des deux n'atteint 2 barres, donc ce n'est pas une différence : en théorie un arbre ignore le -1 (il coupe sur un seuil) alors qu'une somme pondérée s'en laisse tirer, mais ce défaut est trop rare ici pour peser sur l'un ou l'autre : il est établi, mais il ne coûte rien.
Le contrôle trouve 0 ligne(s) dupliquée(s) sur 149894. Un contrôle qui ne trouve rien est un résultat : on a cherché, il n'y a rien à corriger, et on peut le dire avec un chiffre (0 sur 149894).
  ok        les trois prédictions sont écrites avant toute mesure
  ok        deux sc

### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [69]:
import numpy as np

# On garde le modèle et le jeu de contrôle de la mise en place : on ne change QUE la taille du contrôle.
# Pour aller vite, on prédit UNE seule fois sur tout Xval, puis on pioche dans les prédictions.
pred_val = modele.predict(Xval)          # une prédiction par ligne de contrôle
vrai_val = yval.to_numpy()               # les vraies réponses, en tableau numpy
N = len(Xval)                            # nombre total de lignes de contrôle

# p = l'exactitude du modèle sur tout le contrôle (une proportion entre 0 et 1) : sert à la formule
p = (pred_val == vrai_val).mean()

def tirages(n, nb=20, graine=0):
    """Tire `nb` fois `n` lignes de Xval (sans remise dans un même tirage) et rend les `nb` scores en %."""
    rng = np.random.RandomState(graine)                  # générateur de hasard reproductible
    scores = []
    for _ in range(nb):
        idx = rng.choice(N, size=n, replace=False)       # n positions au hasard dans Xval
        scores.append(100 * (pred_val[idx] == vrai_val[idx]).mean())   # exactitude de ce petit tirage
    return scores

# ── 2. Vingt tirages de 1 000 lignes, puis vingt de 5 000 lignes (graines différentes : deux mesures séparées)
ecarts_1000 = tirages(1000, graine=1)
ecarts_5000 = tirages(5000, graine=2)

# ── 3. L'écart-type des 20 scores, pour chaque taille : c'est la barre de bruit OBSERVÉE
# ddof=1 : on estime à partir d'un échantillon de 20 valeurs (écart-type "corrigé").
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))

# ── 4. La barre ANNONCÉE par la formule du cours : racine de p(1-p)/n, en points
formule_1000 = 100 * (p * (1 - p) / 1000) ** 0.5
formule_5000 = 100 * (p * (1 - p) / 5000) ** 0.5

# Le rapport observé / formule : proche de 1 si la formule dit vrai
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000

print(f"n = 1000 : observé {ecart_type_1000:.3f} | formule {formule_1000:.3f} | rapport {rapport_1000:.2f}")
print(f"n = 5000 : observé {ecart_type_5000:.3f} | formule {formule_5000:.3f} | rapport {rapport_5000:.2f}")
print(f"barre(1000) / barre(5000) = {ecart_type_1000 / ecart_type_5000:.2f}   (la formule annonce racine de 5 = {5 ** 0.5:.2f})")

# Précision d'un écart-type estimé avec k valeurs : environ 1 / racine(2(k-1)) en valeur relative
k = 20
precision_estimation = 100 / (2 * (k - 1)) ** 0.5       # en %, environ 16 % pour k = 20

# ── 5. La lecture (modèle de phrase avec tes chiffres : relis-la et reformule avec tes mots)
lecture = (
    f"Quand n est multiplié par 5, la barre diminue : {ecart_type_1000:.3f} point pour 1 000 lignes contre "
    f"{ecart_type_5000:.3f} pour 5 000, soit un facteur {ecart_type_1000 / ecart_type_5000:.2f} "
    f"(la formule annonce racine de 5, environ 2,24). Le rapport observé/formule vaut {rapport_1000:.2f} puis "
    f"{rapport_5000:.2f} : à peu près 1, donc la formule décrit bien ce qui se passe. Mais vingt tirages ne "
    f"suffisent pas à estimer un écart-type avec précision : l'estimation reste incertaine d'environ "
    f"{precision_estimation:.0f} %, donc un rapport de 0,9 ou 1,1 ne prouve ni un accord parfait ni un désaccord. "
    f"Un peu moins de 1 pour 5 000 est normal : on tire dans un contrôle fini de {N} lignes, sans remise."
)
print(lecture)

controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


n = 1000 : observé 1.315 | formule 1.201 | rapport 1.09
n = 5000 : observé 0.471 | formule 0.537 | rapport 0.88
barre(1000) / barre(5000) = 2.79   (la formule annonce racine de 5 = 2.24)
Quand n est multiplié par 5, la barre diminue : 1.315 point pour 1 000 lignes contre 0.471 pour 5 000, soit un facteur 2.79 (la formule annonce racine de 5, environ 2,24). Le rapport observé/formule vaut 1.09 puis 0.88 : à peu près 1, donc la formule décrit bien ce qui se passe. Mais vingt tirages ne suffisent pas à estimer un écart-type avec précision : l'estimation reste incertaine d'environ 16 %, donc un rapport de 0,9 ou 1,1 ne prouve ni un accord parfait ni un désaccord. Un peu moins de 1 pour 5 000 est normal : on tire dans un contrôle fini de 29979 lignes, sans remise.
  ok        vingt tirages à chaque taille
  ok        les scores tirés sont des pourcentages
  ok        les deux barres ont été mesurées SÉPARÉMENT
  ok        les deux barres sont chiffrées
  ok        les deux rapports observé 

### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [70]:
# ── 1. Vingt tirages de 25 000 lignes dans Xval (qui en contient N = 29 979) : presque tout le contrôle
# On réutilise la fonction `tirages` de la Q5 (même méthode, seule la taille change).
ecarts_25000 = tirages(25000, graine=3)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))    # la barre OBSERVÉE (écart-type des 20 scores)

# ── 2. La barre annoncée par la formule du cours, puis le rapport observé / formule
formule_25000 = 100 * (p * (1 - p) / 25000) ** 0.5
rapport_25000 = ecart_type_25000 / formule_25000          # attendu : nettement plus petit que 1

# ── 4. Bonus : la correction de population finie (tirage SANS remise dans N lignes)
# Elle multiplie l'écart-type par racine de (N - n) / (N - 1).
def correction(n):
    return ((N - n) / (N - 1)) ** 0.5

fpc_25000 = correction(25000)
formule_corrigee_25000 = formule_25000 * fpc_25000        # la formule du cours, corrigée
rapport_corrige_25000 = ecart_type_25000 / formule_corrigee_25000   # doit retomber vers 1

# On compare les trois tailles dans un seul tableau (tes trois rapports + ce que la correction prédit)
tableau_rapports = pd.DataFrame({
    "n": [1000, 5000, 25000],
    "rapport observé/formule": [rapport_1000, rapport_5000, rapport_25000],
    "correction attendue": [correction(1000), correction(5000), correction(25000)],
}).round(3)
display(tableau_rapports)
print(f"n = 25000 : observé {ecart_type_25000:.3f} | formule {formule_25000:.3f} | rapport {rapport_25000:.2f}")
print(f"avec la correction ({fpc_25000:.3f}) : formule corrigée {formule_corrigee_25000:.3f} | rapport {rapport_corrige_25000:.2f}")

# ── 3. L'explication (modèle de phrase avec tes chiffres : relis-la et reformule avec tes mots)
pourquoi_la_formule_deborde = (
    f"La formule suppose des tirages indépendants, comme dans un jeu quasi infini ; avec 25 000 lignes tirées "
    f"sans remise parmi {N}, on reprend presque tout le contrôle, donc les tirages se ressemblent et le score "
    f"bouge bien moins que prévu. Les rapports valent {rapport_1000:.2f}, {rapport_5000:.2f} puis "
    f"{rapport_25000:.2f} : entre le deuxième et le troisième, la part du jeu tirée est passée d'environ "
    f"{5000 / N:.0%} à {25000 / N:.0%}, et la correction de population finie ({fpc_25000:.2f}) ramène le "
    f"rapport à {rapport_corrige_25000:.2f}."
)
print(pourquoi_la_formule_deborde)
controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


,n,rapport observé/formule,correction attendue
0,1000,1.094,0.983
1,5000,0.876,0.913
2,25000,0.420,0.408


n = 25000 : observé 0.101 | formule 0.240 | rapport 0.42
avec la correction (0.408) : formule corrigée 0.098 | rapport 1.03
La formule suppose des tirages indépendants, comme dans un jeu quasi infini ; avec 25 000 lignes tirées sans remise parmi 29979, on reprend presque tout le contrôle, donc les tirages se ressemblent et le score bouge bien moins que prévu. Les rapports valent 1.09, 0.88 puis 0.42 : entre le deuxième et le troisième, la part du jeu tirée est passée d'environ 17% à 83%, et la correction de population finie (0.41) ramène le rapport à 1.03.
  ok        vingt tirages de 25 000 lignes
  ok        la barre observée et la formule ont été chiffrées séparément
  ok        le rapport est un nombre positif
  ok        l explication tient en une phrase complète


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [71]:
# Ma prédiction, écrite AVANT toute mesure (à remplacer par ce que TU crois)
pred_pas = (
    "J'attends : à 0,05 un score proche du modèle de base, car chaque arbre corrige peu ; à 0,5 un score proche "
    "avec un arrêt anticipé possible (moins d'arbres construits) ; à 2 un score pire, car chaque arbre sur-corrige "
    "et le modèle s'arrête très tôt. Le pas ne devrait pas être ce qui borne le modèle : c'est le signal des données."
)
# ── Une ligne par pas d'apprentissage (0,05 / 0,5 / 2), tout le reste inchangé ──
reglages = []
for lr in (0.05, 0.5, 2):
    m = HistGradientBoostingClassifier(learning_rate=lr, random_state=0).fit(Xtr, ytr)  # même découpage qu'en mise en place
    sc = 100 * m.score(Xval, yval)                                   # score de validation, en %
    reglages.append({
        "learning_rate": lr,
        "score": round(sc, 2),
        "arbres": int(m.n_iter_),                                    # arbres EFFECTIVEMENT construits (pas ceux demandés)
        "en_barres": round((sc - reference) / erreur_type, 1),       # écart à la référence, en barres de bruit
    })
display(pd.DataFrame(reglages))

# ── Les trois scores sont-ils séparés de plus de deux barres ? (écarts entre scores voisins)
scores_tries = sorted(r["score"] for r in reglages)
ecarts_consecutifs = [(scores_tries[i + 1] - scores_tries[i]) / erreur_type for i in range(2)]
tous_separes = all(e > 2 for e in ecarts_consecutifs)
arbres = [r["arbres"] for r in reglages]
meilleur = max(r["score"] for r in reglages)
meilleur_en_barres = (meilleur - reference) / erreur_type

score_separe = (
    f"Les trois scores sont {', '.join(str(r['score']) for r in reglages)} % ; écarts successifs de "
    f"{ecarts_consecutifs[0]:.1f} et {ecarts_consecutifs[1]:.1f} barre(s) : "
    + ("ils sont séparés de plus de deux barres, la différence est réelle." if tous_separes else
       "au moins un écart reste sous deux barres, donc ils ne sont pas tous séparés.")
)
phrase_arbres = (
    f"Les arbres construits sont {arbres} pour 100 demandés : le modèle s'arrête tout seul quand sa part interne "
    f"cesse de progresser, donc le nombre d'arbres est borné par cet arrêt anticipé et non par ce qu'on demande. "
    if max(arbres) < 100 else
    f"Les arbres construits sont {arbres} : ils atteignent le nombre demandé, l'arrêt anticipé n'a pas joué. "
)
ce_qui_borne = (
    phrase_arbres
    + f"Même avec le meilleur pas, le score ({meilleur:.1f} %) n'est qu'à {meilleur_en_barres:.1f} barre(s) de la "
      f"référence : ce plafond vient surtout du signal que contiennent les données, pas du pas d'apprentissage."
)
print(score_separe)
print(ce_qui_borne)
prevoir("pas d'apprentissage", "trois scores proches (écart sous deux barres), sauf un pas de 2 plus bas",
        "le plafond vient des données, pas du pas ; l'arrêt anticipé réduit le nombre d'arbres construits")

controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


,learning_rate,score,arbres,en_barres
0,0.05,82.50,71,3.4
1,0.50,82.09,15,1.6
2,2.00,82.14,10,1.8


Les trois scores sont 82.5, 82.09, 82.14 % ; écarts successifs de 0.2 et 1.6 barre(s) : au moins un écart reste sous deux barres, donc ils ne sont pas tous séparés.
Les arbres construits sont [71, 15, 10] pour 100 demandés : le modèle s'arrête tout seul quand sa part interne cesse de progresser, donc le nombre d'arbres est borné par cet arrêt anticipé et non par ce qu'on demande. Même avec le meilleur pas, le score (82.5 %) n'est qu'à 3.4 barre(s) de la référence : ce plafond vient surtout du signal que contiennent les données, pas du pas d'apprentissage.


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55
5,pas d'apprentissage,"trois scores proches (écart sous deux barres),...","le plafond vient des données, pas du pas ; l'a...",19:29:05


  ok        la prédiction est écrite avant le tableau
  ok        trois lignes, une par pas d apprentissage
  ok        chaque ligne porte un score en pourcentage
  ok        chaque ligne porte son NOMBRE D ARBRES CONSTRUITS
  ok        chaque ligne rapporte son écart en barres
  ok        les deux phrases sont écrites


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [72]:
# Trois essais de natures différentes : une DONNÉE, un RÉGLAGE, une FAMILLE de modèle.
# Pour chacun : un signe ET une raison, jamais une valeur. (À remplacer par ce que TU attends.)
prevoir("retirer les doublons de colonnes", "gain nul (dans le bruit)",
        "age_mois et montant_acte répètent age et cout_acte : les arbres n'y gagnent rien")
prevoir("300 arbres au lieu de 100", "petit gain positif ou nul",
        "plus d'arbres affinent un peu, mais l'arrêt anticipé borne déjà le nombre construit")
prevoir("forêt aléatoire", "score plus bas ou égal au boosting",
        "une forêt moyenne des arbres indépendants et capte moins de signal fin que le boosting sur cette table")
from sklearn.ensemble import RandomForestClassifier

def score_sur(table, modele_neuf, graine=0):
    """Découpe la table (80/20, stratifié, graine donnée), entraîne un modèle neuf, rend son score de validation."""
    a, b, c, d = train_test_split(table, y, test_size=0.2, stratify=y, random_state=graine)
    return modele_neuf().fit(a, c).score(b, d)

# Les tables de travail de chaque essai
table_sans_doubles = X_sans_features.drop(columns=[c for c in ["age_mois", "montant_acte"] if c in X_sans_features.columns])
table_numerique = pd.get_dummies(X_sans_features, columns=CAT).fillna(0)   # la forêt veut des nombres, sans trous

# nom : (nature, table, fabrique du modèle, description)
ESSAIS_A_MOI = {
    "retirer les doublons de colonnes": ("donnée", table_sans_doubles,
        lambda: HistGradientBoostingClassifier(random_state=0), "retire age_mois et montant_acte"),
    "300 arbres au lieu de 100": ("réglage", X_sans_features,
        lambda: HistGradientBoostingClassifier(max_iter=300, random_state=0), "max_iter = 300"),
    "forêt aléatoire": ("famille de modèle", table_numerique,
        lambda: RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=20, n_jobs=-1, random_state=0),
        "forêt aléatoire de 100 arbres"),
}

# essai(...) prend la mesure LUI-MÊME : l'ordre prédiction -> mesure est prouvé
for nom, (nature, table, modele_neuf, description) in ESSAIS_A_MOI.items():
    essai(nom, lambda t=table, m=modele_neuf: score_sur(t, m), changement=description)

# La table mes_essais : gain par rapport au modèle de la mise en place (score_interne), en barres
lignes = []
for nom, (nature, *_) in ESSAIS_A_MOI.items():
    e = [x for x in ESSAIS if x["hypothèse"] == nom][0]
    gain = (e["mesuré"] - score_interne) / erreur_type
    lignes.append({"essai": nom, "nature": nature, "attendu": e["attendu"], "mesuré": e["mesuré"],
                   "gain_en_barres": round(gain, 2),
                   "verdict": "réel" if abs(gain) >= 2 else "dans le bruit"})   # deux barres : la règle du cours
mes_essais = pd.DataFrame(lignes)
display(mes_essais)
# Phrase écrite AVANT de refaire la mesure (à reformuler avec tes mots)
optimisme = (
    "Mon meilleur essai a été choisi parce qu'il gagne sur cette validation : une part de son avance vient donc "
    "du hasard de ce découpage, et son gain réel est probablement plus petit que celui mesuré."
)

# On reprend le meilleur essai et on le mesure sur un AUTRE découpage (graine 1), avec sa propre base
meilleur = mes_essais.sort_values("gain_en_barres", ascending=False).iloc[0]["essai"]
nature, table, modele_neuf, description = ESSAIS_A_MOI[meilleur]
base2 = score_sur(X_sans_features, lambda: HistGradientBoostingClassifier(random_state=0), graine=1)
essai2 = score_sur(table, modele_neuf, graine=1)
erreur_type2 = 100 * ((base2 * (1 - base2)) / (len(y) * 0.2)) ** 0.5     # la barre de ce second découpage
verifie_par_second_tirage = round(100 * (essai2 - base2) / erreur_type2, 2)
print(meilleur, "→ gain sur le second tirage :", verifie_par_second_tirage, "barre(s)")

controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55
5,pas d'apprentissage,"trois scores proches (écart sous deux barres),...","le plafond vient des données, pas du pas ; l'a...",19:29:05
6,retirer les doublons de colonnes,gain nul (dans le bruit),age_mois et montant_acte répètent age et cout_...,19:29:05


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55
5,pas d'apprentissage,"trois scores proches (écart sous deux barres),...","le plafond vient des données, pas du pas ; l'a...",19:29:05
6,retirer les doublons de colonnes,gain nul (dans le bruit),age_mois et montant_acte répètent age et cout_...,19:29:05
7,300 arbres au lieu de 100,petit gain positif ou nul,"plus d'arbres affinent un peu, mais l'arrêt an...",19:29:05


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55
5,pas d'apprentissage,"trois scores proches (écart sous deux barres),...","le plafond vient des données, pas du pas ; l'a...",19:29:05
6,retirer les doublons de colonnes,gain nul (dans le bruit),age_mois et montant_acte répètent age et cout_...,19:29:05
7,300 arbres au lieu de 100,petit gain positif ou nul,"plus d'arbres affinent un peu, mais l'arrêt an...",19:29:05
8,forêt aléatoire,score plus bas ou égal au boosting,une forêt moyenne des arbres indépendants et c...,19:29:05


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 19:28:48"
2,retirer les doublons de colonnes,retire age_mois et montant_acte,gain nul (dans le bruit),82.47,"oui, à 19:29:05"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 19:28:48"
2,retirer les doublons de colonnes,retire age_mois et montant_acte,gain nul (dans le bruit),82.47,"oui, à 19:29:05"
3,300 arbres au lieu de 100,max_iter = 300,petit gain positif ou nul,82.51,"oui, à 19:29:05"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 19:28:48"
2,retirer les doublons de colonnes,retire age_mois et montant_acte,gain nul (dans le bruit),82.47,"oui, à 19:29:05"
3,300 arbres au lieu de 100,max_iter = 300,petit gain positif ou nul,82.51,"oui, à 19:29:05"
4,forêt aléatoire,forêt aléatoire de 100 arbres,score plus bas ou égal au boosting,81.78,"oui, à 19:29:05"


,essai,nature,attendu,mesuré,gain_en_barres,verdict
0,retirer les doublons de colonnes,donnée,gain nul (dans le bruit),82.47,-0.17,dans le bruit
1,300 arbres au lieu de 100,réglage,petit gain positif ou nul,82.51,0.01,dans le bruit
2,forêt aléatoire,famille de modèle,score plus bas ou égal au boosting,81.78,-3.32,réel


300 arbres au lieu de 100 → gain sur le second tirage : 0.0 barre(s)
  ok        trois essais À VOUS (les amorces ne comptent pas)
  ok        trois natures DIFFÉRENTES
  ok        chaque ligne porte un gain EN BARRES
  ok        chaque verdict est « réel » ou « dans le bruit »
  ok        la réponse est écrite avant le second tirage
  ok        le second tirage est chiffré en barres


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [73]:
# ── 1. L'écart entre les deux familles (arbres vs linéaire), divisé par la barre de bruit
rapport = (score_interne - score_lineaire) / erreur_type          # positif : le linéaire est moins bon

# ── 2. Des coefficients COMPARABLES : on centre-réduit les colonnes (x - moyenne) / écart-type
DOUBLES = ["age", "age_mois", "montant_acte", "cout_acte"]        # colonnes dupliquées : on ne les lit pas
cols_lues = [c for c in X_num.columns if c not in DOUBLES and X_num[c].std() > 0]   # on ignore les colonnes constantes
Z = (X_num[cols_lues] - X_num[cols_lues].mean()) / X_num[cols_lues].std()           # centrées-réduites
modele_z = LogisticRegression(solver="newton-cholesky").fit(Z.loc[Xtr.index], ytr)
coef = pd.Series(modele_z.coef_[0], index=cols_lues).sort_values(key=abs, ascending=False)
display(coef.head(8).round(3))
trois_coefficients = list(coef.index[:3])                         # les trois plus forts, désormais comparables

# ── 3. Les textes : à relire et reformuler avec tes mots
pourquoi_pas_les_dupliquees = (
    "age et age_mois (comme montant_acte et cout_acte) portent la même information : le modèle linéaire se répartit "
    "le poids entre les deux de façon arbitraire, donc chaque coefficient pris seul ne veut plus rien dire."
)
quartier_provoque = (
    "Non : un coefficient de quartier décrit une association sur des données observées, pas une cause ; "
    "le quartier peut simplement refléter autre chose (distance, revenu, délai moyen des rendez-vous)."
)
sens = "en dessous" if rapport > 0 else "au-dessus"
cout_et_rapport = (
    "Le linéaire coûte de la préparation (indicatrices pour les catégories, comblement des trous, mise à l'échelle) "
    "et rapporte des coefficients signés, un par colonne, lisibles ; ici son score est à "
    f"{abs(rapport):.1f} barre(s) {sens} de celui des arbres"
    + (" : moins de deux barres, donc pas une différence nette." if abs(rapport) < 2 else ".")
)
print(rapport, trois_coefficients)
print(cout_et_rapport)

controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


taux_absence_passe      0.331
delai_jours             0.192
rappel_envoye_         -0.178
rappel_envoye_envoyé    0.178
encours_1_mois         -0.132
quartier_Q02           -0.109
quartier_Q01           -0.101
encours_3_mois          0.092
dtype: float64

1.368244722369708 ['taux_absence_passe', 'delai_jours', 'rappel_envoye_']
Le linéaire coûte de la préparation (indicatrices pour les catégories, comblement des trous, mise à l'échelle) et rapporte des coefficients signés, un par colonne, lisibles ; ici son score est à 1.4 barre(s) en dessous de celui des arbres : moins de deux barres, donc pas une différence nette.
  ok        le rapport écart / barre est un nombre fini
  ok        trois coefficients nommés
  ok        aucun coefficient choisi n est une colonne dupliquée
  ok        le refus des dupliquées est argumenté
  ok        la lecture du quartier distingue cause et corrélation
  ok        le coût et le rapport sont écrits


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [74]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # passe_x_delai : le produit « taux d'absence passé × délai du rendez-vous ».
    # Elle découle de mon constat de la Q3 : taux_absence_passe est la meilleure colonne honnête.
    if "taux_absence_passe" in df.columns and "delai_jours" in df.columns:
        passe = pd.to_numeric(df["taux_absence_passe"], errors="coerce")   # en nombre (les trous deviennent NaN)
        delai = pd.to_numeric(df["delai_jours"], errors="coerce")
        df["passe_x_delai"] = passe * delai
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.
# Le constat de la Q3 qui motive la feature (observation EXACTE, telle qu'écrite dans le tableau)
constat_d_origine = "la meilleure colonne du classement filtré est connue après l'issue"

# Prédiction AVANT de mesurer (à remplacer par ce que TU attends)
prevoir("passe_x_delai", "gain nul ou très petit (moins de 2 barres)",
        "les arbres croisent déjà deux colonnes : le produit n'ajoute presque rien")

# remesurer() : la chaîne passe par preparer(), X reçoit la feature UNE fois, modele est mis à jour
score_apres_feature = remesurer()
gain_de_ma_feature = (score_apres_feature - score_interne) / erreur_type     # un gain de zéro est une réponse
print(f"avec ma feature : {score_apres_feature:.2f} % | gain : {gain_de_ma_feature:+.2f} barre(s)")

# ── Le contournement : une table préparée À LA MAIN, avec une étape oubliée ──
def preparer_sans_fillna(brut):
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns], errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            # ICI on a OUBLIÉ .fillna("") : les cases vides restent des NaN au lieu de la catégorie vide
            d[c] = d[c].astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

brut_val = train.loc[Xval.index]                                   # les lignes brutes de la validation
score_propre = 100 * (modele.predict(preparer(brut_val)) == yval).mean()            # la chaîne officielle
score_contourne = 100 * (modele.predict(preparer_sans_fillna(brut_val)) == yval).mean()   # la table faite à la main
ecart_du_contournement = (score_contourne - score_propre) / erreur_type
quelle_etape_oubliee = (
    "J'ai oublié l'étape fillna(\"\") des colonnes catégorielles : les cases vides restent des valeurs manquantes "
    "au lieu d'être la catégorie vide apprise à l'entraînement, et aucune erreur ne prévient."
)
print(score_propre, score_contourne, ecart_du_contournement)

controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55
5,pas d'apprentissage,"trois scores proches (écart sous deux barres),...","le plafond vient des données, pas du pas ; l'a...",19:29:05
6,retirer les doublons de colonnes,gain nul (dans le bruit),age_mois et montant_acte répètent age et cout_...,19:29:05
7,300 arbres au lieu de 100,petit gain positif ou nul,"plus d'arbres affinent un peu, mais l'arrêt an...",19:29:05
8,forêt aléatoire,score plus bas ou égal au boosting,une forêt moyenne des arbres indépendants et c...,19:29:05
9,passe_x_delai,gain nul ou très petit (moins de 2 barres),les arbres croisent déjà deux colonnes : le pr...,19:29:23


avec ma feature : 82.45 % | gain : -0.24 barre(s)
82.45438473598186 82.45104906768071 -0.015202719137457753
  ok        la feature s inspire d un constat de la question 3
  ok        votre feature a enrichi la table
  ok        le gain est chiffré en barres
  ok        l écart du contournement est chiffré en barres
  ok        l étape oubliée est nommée


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [75]:
# 1. Écart de remplissage train / test, colonne par colonne
communes = [c for c in train.columns if c in test.columns and c != CIBLE]
rempl_train = 100 * train[communes].notna().mean()
rempl_test = 100 * test[communes].notna().mean()
ecarts = (rempl_train - rempl_test).abs().sort_values(ascending=False).to_frame("ecart")
display(ecarts.head(5).round(1))

colonne_trouvee = ecarts.index[0]
ecart_remplissage = float(ecarts.iloc[0, 0])
remplissage_du_test = float(rempl_test[colonne_trouvee])
deuxieme = ecarts.index[1]

pourquoi_pas_la_derivee = (
    f"{deuxieme} passe de {rempl_train[deuxieme]:.1f} % de remplissage en train à {rempl_test[deuxieme]:.1f} % en test "
    f"(écart {ecarts.iloc[1, 0]:.1f} points) : elle dérive, mais reste renseignée sur une part importante du test, "
    f"alors que {colonne_trouvee} y disparaît presque : une dérive n'est pas une disparition."
)

# 2. Gain interne, sur mon_train, avec et sans la colonne
X_m, CAT_m, _ = matrice(mon_train)
y_m = mon_train[CIBLE]
col = mon_train[colonne_trouvee]
est_numerique = pd.api.types.is_numeric_dtype(col)

X_avec = X_m.copy()
X_avec[colonne_trouvee] = col if est_numerique else col.fillna("").astype("category")

a, b, c, d = train_test_split(X_avec, y_m, test_size=0.2, stratify=y_m, random_state=0)
modele_avec = HistGradientBoostingClassifier(random_state=0).fit(a, c)
modele_sans = HistGradientBoostingClassifier(random_state=0).fit(a.drop(columns=[colonne_trouvee]), c)

s_avec = 100 * modele_avec.score(b, d)
s_sans = 100 * modele_sans.score(b.drop(columns=[colonne_trouvee]), d)
barre_m = max(100 * ((s_avec / 100) * (1 - s_avec / 100) / len(d)) ** 0.5, 0.01)

gain_interne = s_avec - s_sans
gain_en_barres = gain_interne / barre_m

# 3. La simulation : la colonne rendue manquante dans la validation
taux_de_trous = 1 - remplissage_du_test / 100
rng = np.random.RandomState(0)
masque = rng.rand(len(b)) < taux_de_trous

b_sim = b.copy()
if est_numerique:
    b_sim.loc[masque, colonne_trouvee] = np.nan
else:
    if "" not in b_sim[colonne_trouvee].cat.categories:
        b_sim[colonne_trouvee] = b_sim[colonne_trouvee].cat.add_categories([""])
    b_sim.loc[masque, colonne_trouvee] = ""

score_simule = 100 * modele_avec.score(b_sim, d)
chute_en_barres = (score_simule - s_avec) / barre_m

print(f"{colonne_trouvee} : gain {gain_interne:.2f} pts ({gain_en_barres:.1f} barres) ; simulé {score_simule:.2f} ; chute {chute_en_barres:.1f} barres")

decision_A_ECARTER = (
    f"Écarter {colonne_trouvee} était la bonne décision : son gain interne de {gain_en_barres:.1f} barre(s) tombe de "
    f"{abs(chute_en_barres):.1f} barre(s) quand elle manque comme dans le test, où seulement {remplissage_du_test:.1f} % des lignes "
    f"la portent. Mais J1 l'a écartée sans le dire : la bonne raison est sa disparition en production, pas une impression."
)

controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


,ecart
canal_prise,99.5
motif_absence,0.8
rappel_envoye,0.3
constantes_relevees,0.3
solde_credit,0.0


canal_prise : gain 3.41 pts (13.1 barres) ; simulé 82.51 ; chute -13.0 barres
  ok        l écart de remplissage est un pourcentage
  ok        la colonne nommée est bien celle de VOTRE tri
  ok        la colonne qui dérive est écartée, avec sa raison
  ok        la part du test encore renseignée est un pourcentage
  ok        le gain interne et sa simulation sont chiffrés
  ok        la simulation a été mesurée sur un score à part
  ok        la décision est argumentée, sur les deux questions


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [76]:
# À vous — écrivez votre réponse ici.

# TODO : l'outil, avec les DEUX signes, qui ignore identifiant et cible.
def suspects(df, cible):
    """Rend une table, une ligne par colonne suspecte : nom, écart de remplissage, signe.
       Signe 1 (le VIDE) : plus de 50 points d'écart de remplissage entre les deux groupes.
       Signe 2 (la VALEUR) : une valeur qui ne vit que dans un seul des deux groupes.
       Ajoutez ce que vous jugez utile — mais jamais en lisant la cible par accident."""
    # … (à vous)

def suspects(df, cible, min_lignes=100):
    ignorer = {ID, cible}
    g1 = df[cible] == 1
    lignes = []
    for col in df.columns:
        if col in ignorer:
            continue
        rempli = df[col].notna()
        t1, t0 = 100 * rempli[g1].mean(), 100 * rempli[~g1].mean()
        ecart = abs(t1 - t0)
        if ecart > 50:
            lignes.append({"colonne": col, "ecart_remplissage": round(ecart, 1),
                           "signe": "vide : remplie d'un côté, vide de l'autre"})
            continue
        groupes = df.groupby(col, observed=True)[cible].agg(["size", "min", "max"])
        purs = groupes[(groupes["size"] >= min_lignes) & (groupes["min"] == groupes["max"])]
        if len(purs) > 0:
            lignes.append({"colonne": col, "ecart_remplissage": round(ecart, 1),
                           "signe": f"valeur : {purs.index[0]!r} ne vit que dans un groupe ({int(purs['size'].iloc[0])} lignes)"})
    return pd.DataFrame(lignes)

table_suspects = suspects(train, CIBLE)
display(table_suspects)

VERDICTS = {
    "motif_absence": ("fuite", "Le motif d'absence n'existe que si le patient est absent : il est écrit après l'issue, donc inconnu au moment de décider."),
    "duree_consultation": ("fuite", "La durée d'une consultation n'existe que si le patient est venu : elle est connue après l'issue."),
    "constantes_relevees": ("à discuter", "Des constantes relevées pendant la consultation seraient connues après l'issue, mais il faut vérifier quand elles sont saisies."),
    "canal_prise": ("pas une fuite", "Le canal par lequel le rendez-vous a été pris est connu à la prise du rendez-vous, donc avant l'issue."),
}

lignes = []
for nom in table_suspects["colonne"]:
    v, j = VERDICTS.get(nom, ("à discuter", f"À COMPLÉTER : dire ce que veut dire la valeur de {nom} et quand elle est connue (avant ou après l'issue)."))
    lignes.append({"colonne": nom, "verdict": v, "justification": j})

classement_suspects = pd.DataFrame(lignes).set_index("colonne")
display(classement_suspects)

candidats = []
for c in train.select_dtypes(include="number").columns:
    if c in (ID, CIBLE):
        continue
    neg = train[c][train[c] < 0]
    if 0 < len(neg) < 5000:
        v = neg.value_counts().idxmax()
        candidats.append((c, v, int((train[c] == v).sum())))

print("valeurs négatives rares :", candidats)

col_s, val_s, lignes_porteuses = next(x for x in candidats if x[0] == "age")
sentinelle_trouvee = f"{col_s} = {val_s}"
masque_s = train[col_s] == val_s

taux_sur_ces_lignes = 100 * train.loc[masque_s, CIBLE].mean()
taux_global = 100 * train[CIBLE].mean()
barre_s = 100 * ((taux_global / 100) * (1 - taux_global / 100) / lignes_porteuses) ** 0.5
ecart_s_barres = (taux_sur_ces_lignes - taux_global) / barre_s

verdict_sentinelle = (
    f"{sentinelle_trouvee} sur {lignes_porteuses} lignes : taux d'absence {taux_sur_ces_lignes:.1f} % contre {taux_global:.1f} % "
    f"au global ({ecart_s_barres:+.1f} barre(s)). "
    + ("C'est un défaut de saisie : un code mis à la place d'une mesure manquante, sans lien visible avec l'issue."
       if abs(ecart_s_barres) < 2 else
       "L'écart est net : la valeur impossible semble liée à l'issue, ce qui ressemble à une fuite à discuter.")
)
print(verdict_sentinelle)

controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


,colonne,ecart_remplissage,signe
0,duree_consultation,0.0,valeur : np.int64(-2) ne vit que dans un group...
1,motif_absence,100.0,"vide : remplie d'un côté, vide de l'autre"


,verdict,justification
colonne,,
duree_consultation,fuite,La durée d'une consultation n'existe que si le...
motif_absence,fuite,Le motif d'absence n'existe que si le patient ...


valeurs négatives rares : [('age', np.int64(-1), 46), ('indice_risque_externe', np.float64(-1.07), 3), ('indice_meteo_jour', np.int64(-2), 62), ('age_mois', np.int64(-12), 46)]
age = -1 sur 46 lignes : taux d'absence 10.9 % contre 18.3 % au global (-1.3 barre(s)). C'est un défaut de saisie : un code mis à la place d'une mesure manquante, sans lien visible avec l'issue.
  ok        `suspects` rend une table, pas une liste de mots
  ok        l identifiant et la cible sont exclus
  ok        le signe du VIDE apparaît au moins une fois
  ok        le signe de la VALEUR apparaît au moins une fois
  REVOIR    chaque verdict a été choisi, et le tri n est pas uniforme   ->   la même réponse recopiée sur toutes les lignes n est pas un tri
  ok        chaque colonne signalée est classée, et chaque verdict argumenté
  ok        la sentinelle est nommée, chiffrée, et jugée


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [77]:
# ── 1. L'annonce écrite AVANT les tranches ──
annonce = (
    "J'annonce qu'à délai égal le rappel aide ou ne change rien : l'écart global vient sans doute d'un autre facteur "
    "(le rappel serait envoyé surtout aux rendez-vous lointains). Je changerais d'avis si, dans plusieurs tranches "
    "de délai, le taux d'absence avec rappel dépassait le taux sans rappel de plus de deux barres."
)

# ── 2. Identification des groupes avec et sans rappel ──
COL_RAPPEL = "rappel_envoye"
r = mon_train[COL_RAPPEL]

# Le masque prend exactement la valeur 'envoyé' observée dans la table
avec = r.astype(str).str.lower().str.strip() == "envoyé"
absent = mon_train[CIBLE]

# Écart global
taux_avec_global = 100 * float(absent[avec].mean())
taux_sans_global = 100 * float(absent[~avec].mean())
ecart_global = float(taux_avec_global - taux_sans_global)

# ── 3. Découpage en tranches de délai ──
delai = pd.to_numeric(mon_train["delai_jours"], errors="coerce").fillna(0)
delai_tranches = pd.qcut(delai, 6, duplicates="drop")

tranches = []
tranches_exclues = []

for t in sorted(delai_tranches.dropna().unique()):
    m = delai_tranches == t
    n1 = int((m & avec).sum())
    n0 = int((m & ~avec).sum())
    
    if min(n1, n0) < 500:
        tranches_exclues.append({
            "tranche": str(t),
            "avec_rappel": n1,
            "sans_rappel": n0,
            "raison": "un groupe sous 500 lignes (ou absent) : on ne compare pas"
        })
        continue
    
    p1 = float(absent[m & avec].mean())
    p0 = float(absent[m & ~avec].mean())
    barre = 100 * (p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0) ** 0.5
    ecart = 100 * (p1 - p0)
    
    tranches.append({
        "tranche": str(t),
        "taux_avec": round(100 * p1, 2),
        "taux_sans": round(100 * p0, 2),
        "lignes": min(n1, n0),
        "ecart": round(ecart, 2),
        "barre": round(barre, 2),
        "ecart_en_barres": round(ecart / barre, 1)
    })

display(pd.DataFrame(tranches))
display(pd.DataFrame(tranches_exclues))

# ── 4. Médianes et variable explicative ──
delai_median_avec = float(delai[avec].median()) if avec.sum() > 0 else 0.0
delai_median_sans = float(delai[~avec].median()) if (~avec).sum() > 0 else 0.0

variable_qui_explique = (
    f"delai_jours : le délai médian est de {delai_median_avec:.0f} jours avec rappel contre {delai_median_sans:.0f} jours "
    f"sans rappel ; le rappel est envoyé aux rendez-vous lointains, qui manquent plus souvent de toute façon."
)

# ── 5. Expérience qui trancherait ──
experience_qui_trancherait = (
    "On tirerait au hasard, parmi les rendez-vous à venir, ceux qui reçoivent un rappel et ceux qui n'en reçoivent pas. "
    "On comparerait ensuite le taux d'absence des deux groupes, avec sa barre de bruit. "
    "La comparaison serait honnête parce que le hasard répartit le délai et toutes les autres causes également entre les deux groupes."
)

print(f"Écart global : {ecart_global:+.2f} points")
print(f"Médiane avec : {delai_median_avec} j | Médiane sans : {delai_median_sans} j")

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


,tranche,taux_avec,taux_sans,lignes,ecart,barre,ecart_en_barres
0,"(1.0, 5.0]",24.74,22.80,10735,1.94,0.57,3.4
1,"(5.0, 11.0]",29.54,36.89,675,-7.34,1.92,-3.8
2,"(11.0, 40.0]",30.52,37.26,993,-6.74,1.59,-4.2


,tranche,avec_rappel,sans_rappel,raison
0,"(-0.001, 1.0]",0,46247,un groupe sous 500 lignes (ou absent) : on ne ...


Écart global : +15.16 points
Médiane avec : 11.0 j | Médiane sans : 0.0 j
  ok        l annonce est écrite avant les tranches
  ok        l écart global est chiffré, en points
  ok        les tranches portent leur barre, pas seulement leur écart
  ok        les tranches trop fines de VOTRE échantillon sont mises à part
  ok        chaque tranche comparée porte son écart et son effectif
  ok        les effectifs sont comptés
  ok        la variable qui explique est nommée, et les deux médianes sont chiffrées
  ok        l expérience qui trancherait est décrite en trois lignes


In [78]:
print(mon_train["rappel_envoye"].value_counts(dropna=False))

rappel_envoye
NaN       59745
envoyé    30191
Name: count, dtype: int64


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [79]:
# ── Trois hypothèses : ce qu'elle raconte (avec signe), comment on la teste, ce qui la CONTREDIRAIT ──
# ── 1. Déclarer les trois hypothèses ──
hypothese("passe_chargee",
          "un indicateur « passé chargé » (taux d'absence passé au-dessus de la médiane) aide les arbres à prédire (signe +)",
          "ajouter la feature, mesurer le score sur la même validation, diviser le gain par la barre de bruit",
          "gain inférieur à 2 barres de bruit en valeur absolue, ou score qui baisse")

hypothese("tres_jeune",
          "un patient de moins de 24 mois manque moins souvent : l'indicateur aide à prédire (signe +)",
          "taux d'absence des moins de 24 mois contre les autres, puis gain du modèle avec l'indicateur",
          "taux d'absence des moins de 24 mois à moins de 2 barres de celui des autres")

hypothese("passe_x_delai",
          "le produit taux_absence_passe × delai_jours résume un risque cumulé et aide à prédire (signe +)",
          "remesurer() avec la feature (question 10), gain divisé par la barre de bruit",
          "gain inférieur à 2 barres de bruit en valeur absolue, ou score qui baisse")

# ── 2. Prédiction écrite AVANT la mesure ──
prevoir("passe_chargee", "gain inférieur à 2 barres (hypothèse probablement contredite)",
        "les arbres savent déjà couper taux_absence_passe à la médiane : l'indicateur n'apporte rien de neuf")

# ── 3. Fonction de mesure (renvoie un ratio entre 0 et 1 pour ne pas multiplier deux fois par 100) ──
def mesure_passe_chargee():
    table = X_sans_features.copy()
    passe = pd.to_numeric(table["taux_absence_passe"], errors="coerce")
    table["passe_chargee"] = (passe > passe.median()).astype(int)
    a, b, c, d = train_test_split(table, y, test_size=0.2, stratify=y, random_state=0)
    return HistGradientBoostingClassifier(random_state=0).fit(a, c).score(b, d)

# ── 4. Lancement de l'essai ──
essai("passe_chargee", mesure_passe_chargee,
      changement="ajouter l'indicateur passe_chargee à la table de travail")

# ── 5. Calcul du gain et verdict ──
e = [x for x in ESSAIS if x["hypothèse"] == "passe_chargee"][0]
gain_h = (e["mesuré"] - score_interne) / erreur_type

# Le verdict doit être exactement le mot "contredite" ou "confirmée"
if abs(gain_h) < 2:
    confirmee_ou_contredite = "contredite"
else:
    confirmee_ou_contredite = "confirmée"

suite_donnee = (
    "L'indicateur binaire n'apporte aucun gain mesurable sur le modèle d'arbres, "
    "nous ne conservons pas cette feature dans la table de travail principale."
)

print(f"L'hypothèse est {confirmee_ou_contredite.upper()} (gain de {gain_h:+.2f} barre(s)).")

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


,feature,ce qu'elle raconte,test,contredite si
0,passe_chargee,un indicateur « passé chargé » (taux d'absence...,"ajouter la feature, mesurer le score sur la mê...",gain inférieur à 2 barres de bruit en valeur a...


,feature,ce qu'elle raconte,test,contredite si
0,passe_chargee,un indicateur « passé chargé » (taux d'absence...,"ajouter la feature, mesurer le score sur la mê...",gain inférieur à 2 barres de bruit en valeur a...
1,tres_jeune,un patient de moins de 24 mois manque moins so...,taux d'absence des moins de 24 mois contre les...,taux d'absence des moins de 24 mois à moins de...


,feature,ce qu'elle raconte,test,contredite si
0,passe_chargee,un indicateur « passé chargé » (taux d'absence...,"ajouter la feature, mesurer le score sur la mê...",gain inférieur à 2 barres de bruit en valeur a...
1,tres_jeune,un patient de moins de 24 mois manque moins so...,taux d'absence des moins de 24 mois contre les...,taux d'absence des moins de 24 mois à moins de...
2,passe_x_delai,le produit taux_absence_passe × delai_jours ré...,"remesurer() avec la feature (question 10), gai...",gain inférieur à 2 barres de bruit en valeur a...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:28:48
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:28:48
2,âge à -1 : arbres,écart d'environ 1 barre ou moins,un arbre coupe sur des seuils : -1 tombe dans ...,19:28:55
3,lignes dupliquées,écart d'environ 1 barre,peu de doublons : un petit gonflement du score...,19:28:55
4,nombre rangé en texte,écart de 0 barre,colonne déjà hors de la table de travail : la ...,19:28:55
5,pas d'apprentissage,"trois scores proches (écart sous deux barres),...","le plafond vient des données, pas du pas ; l'a...",19:29:05
6,retirer les doublons de colonnes,gain nul (dans le bruit),age_mois et montant_acte répètent age et cout_...,19:29:05
7,300 arbres au lieu de 100,petit gain positif ou nul,"plus d'arbres affinent un peu, mais l'arrêt an...",19:29:05
8,forêt aléatoire,score plus bas ou égal au boosting,une forêt moyenne des arbres indépendants et c...,19:29:05
9,passe_x_delai,gain nul ou très petit (moins de 2 barres),les arbres croisent déjà deux colonnes : le pr...,19:29:23


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 19:28:48"
2,retirer les doublons de colonnes,retire age_mois et montant_acte,gain nul (dans le bruit),82.47,"oui, à 19:29:05"
3,300 arbres au lieu de 100,max_iter = 300,petit gain positif ou nul,82.51,"oui, à 19:29:05"
4,forêt aléatoire,forêt aléatoire de 100 arbres,score plus bas ou égal au boosting,81.78,"oui, à 19:29:05"
5,passe_chargee,ajouter l'indicateur passe_chargee à la table ...,gain inférieur à 2 barres (hypothèse probablem...,82.51,"oui, à 19:29:29"


L'hypothèse est CONTREDITE (gain de +0.01 barre(s)).
  ok        trois hypothèses
  ok        chacune dit ce qui la contredirait, précisément
  ok        une hypothèse a été testée sous son propre nom
  ok        le verdict est confirmée ou contredite
  ok        la suite donnée est écrite
  ok        les trois tableaux sont remplis avant le rapport


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [80]:
# ── Question 15 : Classification des colonnes de la table de crédit ──

# 1. Dictionnaire des familles (vocabulaire fermé du cours)
familles = {
    'id_client': 'identifiant',
    'date_defaut': 'connue après l\'issue',
    'statut_6_mois': 'connue après l\'issue',
    'montant': 'utilisable',
    'canal': 'utilisable',
    'score_interne': 'utilisable',
}

# 2. Dictionnaire des justifications (une phrase complète par colonne)
justifications = {
    'id_client': (
        "C'est une clé primaire unique qui désigne la ligne mais ne décrit aucun "
        "comportement emprunteur."
    ),
    'date_defaut': (
        "La date n'est renseignée que si l'emprunteur a déjà fait défaut : c'est "
        "une information connue après l'issue de la décision de crédit."
    ),
    'statut_6_mois': (
        "Ce statut est relevé six mois après l'octroi du prêt : il est totalement "
        "inconnu au moment de prendre la décision."
    ),
    'montant': (
        "Le montant demandé est renseigné dès le formulaire de demande, il est "
        "donc disponible au moment de décider."
    ),
    'canal': (
        "Le canal d'entrée de la demande est connu immédiatement lors du dépôt du "
        "dossier avant l'octroi."
    ),
    'score_interne': (
        "Ce score de risque est calculé au moment précis de la demande : il est "
        "parfaitement utilisable pour la décision."
    ),
}

# ── Contrôles intégrés du devoir ──
COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal', 'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition', 'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


  ok        les six colonnes sont classées
  ok        le vocabulaire est celui des quatre familles
  ok        la colonne du défaut n est pas classée « utilisable »
  ok        l identifiant est reconnu comme tel
  ok        le score calculé à la demande reste utilisable
  ok        les six justifications sont écrites, une phrase chacune
  ok        les six colonnes sont classées
  ok        le vocabulaire est celui des quatre familles
  ok        la colonne du défaut n est pas classée « utilisable »
  ok        l identifiant est reconnu comme tel
  ok        le score calculé à la demande reste utilisable
  ok        les six justifications sont écrites, une phrase chacune


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [81]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,la meilleure colonne du classement filtré est ...,duree_consultation : 99.4 points d'écart entre...,"contre 28.5 points pour taux_absence_passe, la...","écarter : elle figure dans A_ECARTER, et cette..."
4,aucun groupe de 500 lignes pour dossier_id,0 groupe de 500 lignes sur 89936 valeurs disti...,contre au moins 500 lignes exigées par groupe ...,écarter : son écart de 100 points est un artef...


HYPOTHÈSES DE FEATURES 



,feature,ce qu'elle raconte,test,contredite si
0,passe_chargee,un indicateur « passé chargé » (taux d'absence...,"ajouter la feature, mesurer le score sur la mê...",gain inférieur à 2 barres de bruit en valeur a...
1,tres_jeune,un patient de moins de 24 mois manque moins so...,taux d'absence des moins de 24 mois contre les...,taux d'absence des moins de 24 mois à moins de...
2,passe_x_delai,le produit taux_absence_passe × delai_jours ré...,"remesurer() avec la feature (question 10), gai...",gain inférieur à 2 barres de bruit en valeur a...


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:28:48"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 19:28:48"
2,retirer les doublons de colonnes,retire age_mois et montant_acte,gain nul (dans le bruit),82.47,"oui, à 19:29:05"
3,300 arbres au lieu de 100,max_iter = 300,petit gain positif ou nul,82.51,"oui, à 19:29:05"
4,forêt aléatoire,forêt aléatoire de 100 arbres,score plus bas ou égal au boosting,81.78,"oui, à 19:29:05"
5,passe_chargee,ajouter l'indicateur passe_chargee à la table ...,gain inférieur à 2 barres (hypothèse probablem...,82.51,"oui, à 19:29:29"


VOS CONTRÔLES 



,contrôle,tient
0,"la part d absents est un pourcentage, et c est...",True
1,"`ma_reference` est un pourcentage, au-dessus d...",True
2,les deux chiffres sont complémentaires,True
3,le prédicteur ne répond que des 0 et des 1,True
4,son score COÏNCIDE avec votre référence,True
...,...,...
95,le vocabulaire est celui des quatre familles,True
96,la colonne du défaut n est pas classée « utili...,True
97,l identifiant est reconnu comme tel,True
98,le score calculé à la demande reste utilisable,True


LA RELECTURE 

  ok        votre identifiant GitHub est renseigné
  ok        cinq constats
  ok        au moins deux constats mesurés par vous
  ok        chaque constat porte un chiffre ET une comparaison
  ok        trois hypothèses
  ok        chaque hypothèse dit ce qui la contredirait
  ok        au moins trois essais à vous — les amorces ne comptent pas
  ok        chaque essai porte un attendu — un signe ET une raison
  ok        les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle
  ok        une hypothèse a été testée sous son propre nom


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
